# Showcase: reasoned plan → colorized image (Colab)

The visual artifact the metrics have been describing. For each image:

| original | greyscale input | DDColor (automatic) | **plan-conditioned** |
|---|---|---|---|

The plan-conditioned column is the whole system: a context prompt → VLM
selection → KB colours → Grounded-SAM masks → colour hints → Control Color.

The centrepiece is `000000002299`, a genuine monochrome 1940s school photo:
there is **no colour ground truth**, so every colour in that output comes
from the era-conditioned plan.

> Runtime → GPU (A100/L4). ~30-40 min including checkpoint downloads.

In [ ]:
!nvidia-smi -L
%cd /content
!test -d chroma-reasoner || git clone https://github.com/tomqi6195/chroma-reasoner.git
!cd chroma-reasoner && git pull && git log --oneline -1
!pip install -q -e chroma-reasoner
# transformers<5 pinned up front: Control Color needs 4.x, and downgrading
# after Grounded-SAM has imported 5.x corrupts the kernel (Phase-2 lesson).
!pip install -q "transformers<5" accelerate pycocotools
import sys; sys.path.insert(0, '/content/chroma-reasoner/src')
import transformers; print('transformers', transformers.__version__)

## 1. Plans + images

In [ ]:
import glob, json, os, shutil, urllib.request
import cv2
import numpy as np

os.makedirs('plans/reasoned', exist_ok=True)
for src in glob.glob('chroma-reasoner/plans/reasoned/*.json'):
    shutil.copy(src, 'plans/reasoned/')

UPLOAD_PLANS = False   # True to upload a zip of your local plans/reasoned
if UPLOAD_PLANS:
    from google.colab import files
    for name in files.upload():
        !unzip -oq {name}

PLAN_PATHS = sorted(glob.glob('plans/reasoned/*.json'))
IMAGE_IDS = [os.path.basename(p)[:-5] for p in PLAN_PATHS]
print(len(IMAGE_IDS), 'plans:', IMAGE_IDS)

os.makedirs('images', exist_ok=True); os.makedirs('gray', exist_ok=True)
for iid in IMAGE_IDS:
    dest = f'images/{iid}.jpg'
    if not os.path.exists(dest):
        urllib.request.urlretrieve(f'http://images.cocodataset.org/val2017/{iid}.jpg', dest)
    img = cv2.imread(dest)
    cv2.imwrite(f'gray/{iid}.png', cv2.cvtColor(img, cv2.COLOR_BGR2LAB)[:, :, 0])
print('images ready')

## 2. Ground the plans (Grounding DINO + SAM), then free VRAM

In [ ]:
import torch
from PIL import Image
from transformers import (AutoModelForZeroShotObjectDetection, AutoProcessor,
                          SamModel, SamProcessor)
from chroma_reasoner.plan import load_plan
from chroma_reasoner.plan.masks import region_key, save_mask

device = 'cuda'
dino_proc = AutoProcessor.from_pretrained('IDEA-Research/grounding-dino-base')
dino = AutoModelForZeroShotObjectDetection.from_pretrained('IDEA-Research/grounding-dino-base').to(device).eval()
sam_proc = SamProcessor.from_pretrained('facebook/sam-vit-huge')
sam = SamModel.from_pretrained('facebook/sam-vit-huge').to(device).eval()

@torch.no_grad()
def phrase_to_mask(image_pil, phrase):
    text = phrase.lower().rstrip('.') + '.'
    inputs = dino_proc(images=image_pil, text=text, return_tensors='pt').to(device)
    res = dino_proc.post_process_grounded_object_detection(
        dino(**inputs), inputs.input_ids, threshold=0.25, text_threshold=0.2,
        target_sizes=[image_pil.size[::-1]])[0]
    if len(res['boxes']) == 0:
        return None
    box = res['boxes'][res['scores'].argmax()].tolist()
    s_in = sam_proc(image_pil, input_boxes=[[box]], return_tensors='pt').to(device)
    s_out = sam(**s_in)
    masks = sam_proc.image_processor.post_process_masks(
        s_out.pred_masks.cpu(), s_in['original_sizes'].cpu(), s_in['reshaped_input_sizes'].cpu())[0][0]
    return masks[s_out.iou_scores.cpu()[0, 0].argmax()].numpy().astype(bool)

for plan_path in PLAN_PATHS:
    plan = load_plan(plan_path)
    iid = plan['image_id']
    gray = cv2.imread(f'gray/{iid}.png', cv2.IMREAD_GRAYSCALE)
    pil = Image.fromarray(np.stack([gray] * 3, axis=-1))
    hits = 0
    for region in plan['regions']:
        mask = phrase_to_mask(pil, region['grounding_phrase'])
        if mask is not None:
            save_mask(mask, 'masks_show', iid, region); hits += 1
    print(f'{iid}: {hits}/{len(plan["regions"])} masks')

import gc
del dino, sam
gc.collect(); torch.cuda.empty_cache()
print(f'free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

## 3. Colour hints from plan + masks

In [ ]:
import matplotlib.pyplot as plt
from chroma_reasoner.plan.hints import make_hint_image, render_naive
from chroma_reasoner.plan.masks import load_masks

os.makedirs('hints', exist_ok=True)
os.makedirs('results/naive', exist_ok=True)
USABLE = []
for plan_path in PLAN_PATHS:
    plan = load_plan(plan_path)
    iid = plan['image_id']
    gray = cv2.imread(f'gray/{iid}.png', cv2.IMREAD_GRAYSCALE)
    masks = load_masks('masks_show', iid, plan, shape=gray.shape, allow_missing=True)
    if not masks:
        print(f'{iid}: no masks, skipping'); continue
    gray_rgb = np.stack([gray] * 3, axis=-1)
    hint = make_hint_image(gray_rgb, masks, plan, erosion=0.15)
    cv2.imwrite(f'hints/{iid}_input.png', cv2.cvtColor(gray_rgb, cv2.COLOR_RGB2BGR))
    cv2.imwrite(f'hints/{iid}_hint.png', cv2.cvtColor(hint, cv2.COLOR_RGB2BGR))
    cv2.imwrite(f'results/naive/{iid}.png',
                cv2.cvtColor(render_naive(gray, masks, plan), cv2.COLOR_RGB2BGR))
    USABLE.append(iid)
print('usable:', USABLE)

## 4. DDColor — the automatic baseline (no plan, no prompt)

In [ ]:
!test -d DDColor || git clone --depth 1 https://github.com/piddnad/DDColor.git
sys.path.insert(0, '/content/DDColor')
from ddcolor import ColorizationPipeline, DDColor
from huggingface_hub import PyTorchModelHubMixin

class DDColorHF(DDColor, PyTorchModelHubMixin):
    def __init__(self, config=None, **kw):
        super().__init__(**({**config, **kw} if isinstance(config, dict) else kw))

ddc = DDColorHF.from_pretrained('piddnad/ddcolor_paper_tiny').to('cuda').eval()
pipeline = ColorizationPipeline(ddc, input_size=512, device=torch.device('cuda'))

os.makedirs('results/ddcolor', exist_ok=True)
for iid in USABLE:
    out = pipeline.process(cv2.imread(f'hints/{iid}_input.png'))
    cv2.imwrite(f'results/ddcolor/{iid}.png', out)
print('DDColor done')

del ddc, pipeline
gc.collect(); torch.cuda.empty_cache()

## 5. Control Color setup

Checkpoints: https://drive.google.com/drive/folders/1lgqstNwrMCzymowRsbGM-4hk0-7L-eOT
If gdown is quota-blocked, Make-a-copy both files into your Drive and run the
fallback cell (same routine as the L-CAD weights).

In [ ]:
%cd /content
!test -d Control-Color || git clone https://github.com/ZhexinLiang/Control-Color.git
!pip install -q pytorch-lightning==1.9.5 omegaconf einops kornia open-clip-torch taming-transformers-rom1504 gradio safetensors torchviz
!mkdir -p Control-Color/pretrained_models
!gdown --folder https://drive.google.com/drive/folders/1lgqstNwrMCzymowRsbGM-4hk0-7L-eOT -O Control-Color/pretrained_models --remaining-ok
!ls -lh Control-Color/pretrained_models

In [ ]:
# Drive fallback if gdown was quota-blocked
NEED = ['main_model.ckpt', 'content-guided_deformable_vae.ckpt']
if not all(os.path.exists(f'Control-Color/pretrained_models/{n}') for n in NEED):
    from google.colab import drive
    drive.mount('/content/drive')
    for name in NEED:
        dest = f'Control-Color/pretrained_models/{name}'
        if not os.path.exists(dest):
            hits = glob.glob(f'/content/drive/MyDrive/**/*{name}', recursive=True)
            print(name, '->', hits)
            assert hits, f'no copy of {name} in Drive yet'
            shutil.copy(hits[0], dest)
!ls -lh Control-Color/pretrained_models

In [ ]:
# Patch the author-machine paths, stub the two libraries we never use, and
# strip everything gradio-dependent from test.py.
!grep -rln "/data/pretrained/clip-vit-large-patch14" Control-Color | xargs -r sed -i "s|/data/pretrained/clip-vit-large-patch14/*|openai/clip-vit-large-patch14|g"

import types

# lavis: only used to auto-caption when the prompt is empty; we always pass one.
fake_models = types.ModuleType('lavis.models')
fake_models.load_model_and_preprocess = lambda **kw: (None, {'eval': None}, None)
fake_lavis = types.ModuleType('lavis'); fake_lavis.models = fake_models
sys.modules['lavis'] = fake_lavis; sys.modules['lavis.models'] = fake_models

# gradio: test.py imports it at module level, but every actual use is in the
# UI code we cut below. Stubbing beats version-pinning — real gradio drags in
# a huggingface_hub version that fights transformers<5.
class _Progress:
    def __init__(self, *a, **kw): pass
    def __call__(self, *a, **kw): return None
    def tqdm(self, x, *a, **kw): return x
fake_gr = types.ModuleType('gradio')
fake_gr.Progress = _Progress
fake_gr.Blocks = _Progress
sys.modules['gradio'] = fake_gr

# Cut at the FIRST gradio-dependent definition, not just at gr.Blocks:
# get_grayscale_img() has `progress=gr.Progress(...)` as a default argument,
# evaluated at import time. process() is fully defined before that point.
src = open('/content/Control-Color/test.py').read()
markers = ['def get_grayscale_img', 'gr.Progress(', 'block = gr.Blocks', 'gr.Blocks(']
found = [src.find(m) for m in markers]
cut = min([p for p in found if p > 0], default=-1)
assert cut > 0, 'could not find the gradio section'
assert 'def process(' in src[:cut], 'cut too early - process() would be lost'
open('/content/Control-Color/test_api.py', 'w').write(src[:cut])
print(f'kept {src[:cut].count(chr(10))} lines, dropped the gradio UI')

%cd /content/Control-Color
if '.' not in sys.path:
    sys.path.insert(0, '.')
from test_api import process
%cd /content
print('process() ready')

## 6. Render + compare

In [ ]:
from chroma_reasoner.plan.adherence import evaluate_adherence
from chroma_reasoner.plan.hints import global_prompt_terms

os.makedirs('results/plan_conditioned', exist_ok=True)
%cd /content/Control-Color
for iid in USABLE:
    plan = load_plan(f'/content/plans/reasoned/{iid}.json')
    # The plan's context prompt AND its global block drive the diffusion model.
    # Per-region hints only constrain masked pixels; the global block is the
    # only lever over everything else, and unmasked pixels are exactly where
    # the first showcase run produced neon 1940s cardigans.
    g_pos, g_neg = global_prompt_terms(plan)
    prompt = plan.get('prompt') or 'a photograph'
    a_prompt = ', '.join(x for x in ['best quality, detailed, real', g_pos] if x)
    n_prompt = ', '.join(x for x in [
        'a black and white photo, longbody, lowres, bad anatomy, extra digit, '
        'cropped, worst quality, low quality', g_neg] if x)
    print(f'{iid}: prompt={prompt!r}')
    if g_pos:
        print(f'   global+: {g_pos}\n   global-: {g_neg}')

    inp = cv2.cvtColor(cv2.imread(f'/content/hints/{iid}_input.png'), cv2.COLOR_BGR2RGB)
    hint = cv2.cvtColor(cv2.imread(f'/content/hints/{iid}_hint.png'), cv2.COLOR_BGR2RGB)
    outs = process(using_deformable_vae=False, change_according_to_strokes=True,
                   iterative_editing=False, input_image=inp, hint_image=hint,
                   prompt=prompt, a_prompt=a_prompt, n_prompt=n_prompt,
                   num_samples=1, image_resolution=512, ddim_steps=20,
                   guess_mode=False, strength=1.0, scale=7.0,
                   sag_scale=0.05, SAG_influence_step=600, seed=42, eta=0.0)
    out = outs[0] if isinstance(outs, list) else outs
    H, W = inp.shape[:2]
    full = cv2.resize(out, (W, H), interpolation=cv2.INTER_LANCZOS4)
    cv2.imwrite(f'/content/results/plan_conditioned/{iid}.png',
                cv2.cvtColor(full, cv2.COLOR_RGB2BGR))
%cd /content

In [ ]:
os.makedirs('results/figures', exist_ok=True)
for iid in USABLE:
    plan = load_plan(f'plans/reasoned/{iid}.json')
    gray = cv2.imread(f'gray/{iid}.png', cv2.IMREAD_GRAYSCALE)
    masks = load_masks('masks_show', iid, plan, shape=gray.shape, allow_missing=True)
    panels = [
        ('original', cv2.cvtColor(cv2.imread(f'images/{iid}.jpg'), cv2.COLOR_BGR2RGB)),
        ('greyscale input', np.stack([gray] * 3, axis=-1)),
        ('DDColor (automatic)', cv2.cvtColor(cv2.imread(f'results/ddcolor/{iid}.png'), cv2.COLOR_BGR2RGB)),
        ('plan-conditioned', cv2.cvtColor(cv2.imread(f'results/plan_conditioned/{iid}.png'), cv2.COLOR_BGR2RGB)),
    ]
    fig, axes = plt.subplots(1, 4, figsize=(22, 6))
    for ax, (title, img) in zip(axes, panels):
        ax.imshow(img); ax.set_title(title, fontsize=13); ax.axis('off')
    fig.suptitle(f"{iid} — prompt: {plan.get('prompt') or '(none)'}", fontsize=15)
    fig.tight_layout()
    fig.savefig(f'results/figures/{iid}.png', dpi=110, bbox_inches='tight')
    plt.show()

    rgb = cv2.cvtColor(cv2.imread(f'results/plan_conditioned/{iid}.png'), cv2.COLOR_BGR2RGB)
    rep = evaluate_adherence(rgb, masks, plan)
    print(f"   adherence: mean dE {rep['mean_delta_e']} | "
          f"pass {rep['n_pass']}/{rep['n_regions']}")
    for r in rep['regions']:
        if 'delta_e' in r:
            print(f"      {r['region']:>16} dE={r['delta_e']:<6} {'ok' if r['pass'] else 'MISS'}")

In [ ]:
!zip -rq showcase_outputs.zip results/figures results/plan_conditioned results/ddcolor results/naive hints masks_show
from google.colab import files
files.download('showcase_outputs.zip')